# Validación sintética Spark/Delta — cotizaciones Bronze

Ejecutar completo únicamente después de autorización de compute, esquema y storage.
No forma parte de los jobs. No hay limpieza automática. Cada caso usa un esquema
nuevo bajo el prefijo aprobado y storage bajo el run_id; no usar rutas operativas.

CA01–14 se comprueban con datos y Delta reales. Las inyecciones fallan después de
escrituras reales. La suite no simula el runtime. Una reserva de driver abortado
permanece fail-closed: requiere reconciliación operativa antes de liberarla.


In [ ]:
import csv
import io
import json
import os
import re
import sys
import uuid
import platform
import posixpath
from urllib.parse import urlsplit, unquote
from datetime import date
from decimal import Decimal
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

def validate_test_scope(catalog, data_root, schema_prefix):
    expected_catalog = "crea_cotizaciones_test_20261020_01a15a7c"
    expected_root = "abfss://demo@demodldb.dfs.core.windows.net/_tests/cotizaciones/20261020_01a15a7c/data"
    if catalog.lower().startswith("naturapet") or catalog != expected_catalog:
        raise ValueError("UNAUTHORIZED_TEST_CATALOG")
    if data_root != expected_root:
        raise ValueError("UNAUTHORIZED_TEST_DATA_ROOT")
    if not re.fullmatch(r"hu_cotizaciones_test_[a-z0-9_]+", schema_prefix):
        raise ValueError("INVALID_TEST_SCHEMA_PREFIX")
    return catalog, data_root

def assert_test_path(path, data_root):
    expected = urlsplit(data_root)
    actual = urlsplit(path)
    canonical_path = actual.path[:-1] if actual.path.endswith("/") else actual.path
    if (actual.scheme != expected.scheme or actual.netloc != expected.netloc or
        actual.query or actual.fragment or unquote(actual.path) != actual.path or
        posixpath.normpath(canonical_path) != canonical_path or
        not (canonical_path == expected.path or canonical_path.startswith(expected.path + "/")) or
        not path.startswith(expected.scheme + "://") or "?" in path or "#" in path or
        "'" in path or "\\" in path):
        raise ValueError("PATH_OUTSIDE_APPROVED_TEST_DATA_ROOT")
    return path

def prepare_test_leaf(fs, leaf, data_root):
    assert_test_path(leaf, data_root)
    if leaf.rsplit("/", 1)[0] != data_root:
        raise ValueError("EXPECTED_DIRECT_RUN_LEAF")
    try:
        fs.ls(leaf)
    except Exception as error:
        missing = isinstance(error, FileNotFoundError) or re.search(
            r"(?:^|[\s:])(?:java\.io\.)?FileNotFoundException\b", str(error))
        if not missing:
            raise
    else:
        raise FileExistsError("TEST_RUN_LEAF_ALREADY_EXISTS")
    if not fs.mkdirs(leaf):
        raise RuntimeError("TEST_RUN_LEAF_CREATE_FAILED")

def widget(name):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, "")
        return ""

assert widget("allow_synthetic_writes") == "true", "Falta autorización de escrituras sintéticas"
schema_prefix = widget("approved_schema_prefix")
test_catalog, storage_root = validate_test_scope(
    widget("approved_catalog"), widget("approved_storage_root"), schema_prefix)
compute_reference = widget("approved_compute_reference")
assert compute_reference == "1020-210854-9y22c5h7", "Compute diferente del aprobado"
checkout_path = widget("candidate_checkout_path")
assert checkout_path == "/Workspace/Users/srinconr@creasistemas.com/crea_tests/20261020_01a15a7c/checkout"
assert Path(checkout_path, "src/common/cotizaciones_ingestion.py").is_file()
sys.path.insert(0, checkout_path)
from src.common.config import get_config, cotizaciones_paths
from src.common.cotizaciones_contract import FIELDS, FIELD_TYPES, normalize_record
from src.common.cotizaciones_ingestion import CotizacionesEngine, SparkDeltaStore, archive_receipt, _spark_types
from src.common.notebook_runner import run_bronze_load, run_bronze_archive
from pyspark.sql import functions as F, types as T

assert spark.conf.get("spark.databricks.workspaceUrl") == "adb-7405606739630987.7.azuredatabricks.net"
assert spark.sparkContext is not None
assert spark.conf.get("spark.databricks.clusterUsageTags.clusterId") == compute_reference
identity = spark.sql("SELECT current_user() AS identity").first().identity
assert identity == "srinconr@creasistemas.com"
spark.conf.set("spark.sql.shuffle.partitions", "4")  # Only this isolated synthetic session.
runtime = os.environ.get("DATABRICKS_RUNTIME_VERSION", "")
assert runtime, "Registrar DBR real"
run_id = widget("test_run_id") or uuid.uuid4().hex[:12]
assert re.fullmatch(r"[a-z0-9_]{8,40}", run_id), "run_id inválido"
root = assert_test_path(storage_root + "/" + run_id, storage_root)
raw_fs = dbutils.fs
print(json.dumps({"inventory": "PLANNED", "kind": "directory", "path": root}))
# Never mkdir/ls the /data ancestor: it contains registered UC external tables.
prepare_test_leaf(raw_fs, root, storage_root)
print(json.dumps({"inventory": "CREATED", "kind": "directory", "path": root}))
inventory_path = root + "/_inventory.json"
inventory = []
planned_delta_paths = set()
planned_tables = set()

def inventory_event(kind, name, state, **details):
    inventory.append({"kind": kind, "name": name, "state": state, **details})
    raw_fs.put(inventory_path, json.dumps(inventory, ensure_ascii=False), True)
    print(json.dumps({"inventory": inventory[-1]}, ensure_ascii=False))

inventory_event("directory", root, "CREATED", inventory_path=inventory_path)

class GuardedFS:
    def ls(self, path):
        return raw_fs.ls(assert_test_path(path, storage_root))

    def _write(self, operation, paths, *args):
        for path in paths:
            assert_test_path(path, storage_root)
            inventory_event("filesystem", path, "BEFORE_" + operation)
        try:
            result = getattr(raw_fs, operation)(*paths, *args)
        except Exception as error:
            for path in paths:
                inventory_event("filesystem", path, "FAILED_" + operation, error_type=type(error).__name__)
            raise
        for path in paths:
            inventory_event("filesystem", path, "AFTER_" + operation)
        return result

    def mkdirs(self, path):
        return self._write("mkdirs", (path,))

    def put(self, path, contents, overwrite=False):
        return self._write("put", (path,), contents, overwrite)

    def cp(self, source, target):
        return self._write("cp", (source, target))

    def mv(self, source, target):
        return self._write("mv", (source, target))

    def rm(self, path, recurse=False):
        assert recurse is False, "Sin limpieza recursiva automática"
        return self._write("rm", (path,), recurse)

dbutils = SimpleNamespace(fs=GuardedFS(), jobs=dbutils.jobs, widgets=dbutils.widgets, notebook=dbutils.notebook)
metadata = {
    "run_id": run_id, "profile": "CREA_DEV", "catalog": test_catalog,
    "compute": compute_reference, "host": spark.conf.get("spark.databricks.workspaceUrl"),
    "DBR": runtime, "Spark": spark.version, "Python": platform.python_version(),
    "Delta": "integrada en DBR " + runtime, "storage_root": root,
    "identity": identity,
    "inventory_path": inventory_path, "checkout": checkout_path,
    "shuffle_partitions": int(spark.conf.get("spark.sql.shuffle.partitions")),
}
assert metadata["identity"] == "srinconr@creasistemas.com"
inventory_event("runtime", compute_reference, "RUNNING", metadata=metadata)
report = {}

def finalize_inventory():
    for path in sorted(planned_delta_paths):
        try:
            raw_fs.ls(assert_test_path(path, storage_root))
            state = "EXISTS"
        except Exception as error:
            state = "UNAVAILABLE_" + type(error).__name__
        inventory_event("delta_path", path, state)
    for table in sorted(planned_tables):
        inventory_event("table", table, "EXISTS" if spark.catalog.tableExists(table) else "NOT_CREATED")
    print(json.dumps({"inventory_path": inventory_path, "resources": len(inventory)}))


In [ ]:
def valid(**changes):
    base = dict(zip(FIELDS, ["Q001", "20260203", "2026-02-03", "2026-02",
        " T001 ", "001", "C1", "P1", "G1", "2", "100.00", "0", "0",
        "200", "0.19", "38", "238", "COP", "Aceptada", "30"]))
    return base | changes

def as_csv(rows, extra=()):
    stream = io.StringIO(newline="")
    writer = csv.DictWriter(stream, fieldnames=[*FIELDS, *extra])
    writer.writeheader()
    writer.writerows(rows)
    return stream.getvalue()

def config_for(label, year="2026", area="comercial"):
    schema = schema_prefix + "_" + run_id + "_" + label
    assert re.fullmatch(r"[a-z0-9_]+", schema)
    assert not spark.catalog.databaseExists(test_catalog + "." + schema), "Namespace ya existe"
    inventory_event("schema", test_catalog + "." + schema, "BEFORE_CREATE")
    spark.sql("CREATE SCHEMA " + test_catalog + "." + schema)
    inventory_event("schema", test_catalog + "." + schema, "CREATED")
    base = root + "/" + label
    config = get_config(area=area) | {
        "env": "dev", "catalog": test_catalog, "area": area, "bronze_schema": schema,
        "data_year": year, "raw_year_path": base + "/raw/" + year,
        "raw_area_root": base + "/raw", "bronze_area_root": base + "/bronze",
        "bronze_year_root": base + "/bronze/" + year,
        "historic_area_root": base + "/historic", "historic_year_root": base + "/historic/" + year,
        "audit_table_path": base + "/generic_control", "audit_table_name": "bronze_file_load_audit",
        "data_dictionary_path": base + "/dictionary.csv",
    }
    for key in ("raw_year_path", "raw_area_root", "bronze_area_root", "bronze_year_root", "historic_area_root", "historic_year_root", "audit_table_path", "data_dictionary_path"):
        assert_test_path(config[key], storage_root)
    for name, path in cotizaciones_paths(config).items():
        if name == "target_table":
            planned_tables.add(path)
            inventory_event("table", path, "PLANNED")
        else:
            assert_test_path(path, storage_root)
            planned_delta_paths.add(path)
            inventory_event("delta_path", path, "PLANNED")
    return config

def put(config, rows=None, month="02", name=None, content=None, alternate=False):
    name = name or "fact_cotizaciones_" + month + ".csv"
    base = config["raw_year_path"]
    if alternate:
        base = config["raw_area_root"] + "/alternate/" + config["data_year"]
    source = {"source_path": base + "/" + month + "/" + name,
        "source_file_name": name, "source_month": month,
        "source_format": "csv", "load_mode": "incremental"}
    dbutils.fs.mkdirs(source["source_path"].rsplit("/", 1)[0])
    dbutils.fs.put(source["source_path"], content if content is not None else as_csv(rows), True)
    return source

def engine(config, store=None):
    store = store or SparkDeltaStore(spark, dbutils, config)
    return store, CotizacionesEngine(store, config | cotizaciones_paths(config))

def read(config, name):
    return spark.read.format("delta").load(cotizaciones_paths(config)[name])

def expect_failure(action, token=None):
    try:
        action()
    except Exception as error:
        if token:
            assert token in str(error), (token, type(error).__name__, str(error)[:200])
        return
    raise AssertionError("Se esperaba fallo")

def audit_values(config):
    return [json.loads(r.payload) for r in read(config, "file_audit")
        .filter(F.col("file_id") != "__ENTITY__").toLocalIterator()]

def assert_balanced(config, audit):
    store = SparkDeltaStore(spark, dbutils, config)
    store.reconcile(audit)
    m = audit["metrics"]
    assert m["read"] == m["accepted"] + m["rejected"] + m["consolidated"]
    assert audit["status"] == "SUCCESS"

class FaultStore(SparkDeltaStore):
    def __init__(self, *args, fail_after=None, fail_log=False, **kwargs):
        super().__init__(*args, **kwargs)
        self.fail_after, self.fail_log = fail_after, fail_log

    def checkpoint(self, stage):
        if self.fail_after == stage:
            self.fail_after = None
            raise RuntimeError("INJECTED_" + stage)

    def log_failure(self, audit, stage, error):
        if self.fail_log:
            raise RuntimeError("INJECTED_LOG_FAILED")
        return super().log_failure(audit, stage, error)


In [ ]:
def check_contract_and_evidence():
    config = config_for("contract")
    store, run = engine(config)
    rows = [
        valid(producto_id='ñ, "gato" 🐾\nsegunda línea'),
        valid(cotizacion_id="Q2", fecha_cotizacion="03/02/2026"),
        valid(producto_id='ñ, "gato" 🐾\nsegunda línea', cotizacion_id=" Q001 "),
        valid(cotizacion_id=""),
        valid(cotizacion_id="Q3", cantidad="cinco", fecha_id=""),
    ]
    source = put(config, rows)
    audit = run.ingest(source)
    assert audit["metrics"] == {"read": 5, "accepted": 2, "rejected": 2, "consolidated": 1}
    assert_balanced(config, audit)
    main, evidence, quarantine = (read(config, name) for name in ("main", "record_evidence", "quarantine"))
    assert main.count() == 2 and evidence.count() == 5 and quarantine.count() == 2
    assert main.filter(F.col("cotizacion_id") == "Q001").first().producto_id == rows[0]["producto_id"]
    assert quarantine.filter(F.size("errors") >= 2).count() >= 1
    assert evidence.select("_np_record_id").distinct().count() == 5
    # Identity and values must survive repartitioning.
    parsed = store.prepare(audit)
    assert evidence.exceptAll(parsed.repartition(3)).count() == 0
    assert parsed.repartition(2).exceptAll(evidence).count() == 0
    parsed.unpersist()
    assert store.checksum(audit["original_path"]) == audit["checksum"]
    assert evidence.filter(F.col("category") == "CONSOLIDATED").select("payload").first().payload
    assert evidence.filter(F.col("conversions").contains("TRIM_NULL")).count() >= 1
    _, _, business, _, _ = _spark_types()
    expected_schema = T.StructType([T.StructField(f.name, f.dataType, f.name != "cotizacion_id") for f in business])
    assert main.select(*FIELDS).schema == expected_schema
    expected = spark.createDataFrame([normalize_record(rows[0], "2026-02")["values"],
                                     normalize_record(rows[1], "2026-02")["values"]], expected_schema)
    actual = main.select(*FIELDS)
    assert actual.exceptAll(expected).count() == expected.exceptAll(actual).count() == 0
    detail = spark.sql("DESCRIBE DETAIL " + cotizaciones_paths(config)["target_table"]).first()
    assert detail.location.rstrip("/") == cotizaciones_paths(config)["main"]
    assert detail.partitionColumns == ["_np_source_year", "_np_source_month"]
    assert spark.sql("DESCRIBE EXTENDED " + cotizaciones_paths(config)["target_table"]).filter(
        (F.col("col_name") == "Type") & (F.col("data_type") == "EXTERNAL")).count() == 1
    listed = {r.tableName for r in spark.sql("SHOW TABLES IN " + test_catalog + "." + config["bronze_schema"]).toLocalIterator()}
    assert listed == {"fact_cotizaciones"}
    report.update({ca: "passed" for ca in ["CA01", "CA02", "CA03", "CA04", "CA05", "CA11"]})
    for key in ("raw_year_path", "raw_area_root", "bronze_area_root", "bronze_year_root", "historic_area_root", "historic_year_root", "audit_table_path", "data_dictionary_path"):
        assert_test_path(config[key], storage_root)
    for name, path in cotizaciones_paths(config).items():
        if name == "target_table":
            planned_tables.add(path)
            inventory_event("table", path, "PLANNED")
        else:
            assert_test_path(path, storage_root)
            planned_delta_paths.add(path)
            inventory_event("delta_path", path, "PLANNED")
    return config

def check_quality():
    config = config_for("quality")
    store, run = engine(config)
    invalid = [
        ("fecha_cotizacion", "31/02/2026"), ("fecha_cotizacion", ""),
        ("fecha_cotizacion", "2026-01-03"), ("fecha_id", "20260204"), ("fecha_id", ""),
        ("mes_carga", "2026-02-01"), ("mes_carga", ""),
        ("cantidad", "cinco"), ("cantidad", "2147483648"), ("cantidad", "1.5"),
        ("precio_unitario", "125.000,50"), ("descuento_pct", "19%"),
        ("precio_unitario", "10000000000000000"), ("iva_tasa", "0.00001"),
    ]
    rows = [valid(cotizacion_id="BAD" + str(i), **{f: value}) for i, (f, value) in enumerate(invalid)]
    rows += [valid(cotizacion_id="ALERT", cantidad="0", vigencia_dias="-1", descuento_pct="1.1",
                   moneda="USD", estado_cotizacion="Pendiente"),
             valid(cotizacion_id="NULL", descuento_valor="", iva_tasa=""),
             valid(cotizacion_id="T01", valor_total_cotizacion="238.01"),
             valid(cotizacion_id="T02", valor_total_cotizacion="238.02"),
             valid(cotizacion_id="HALF", cantidad="1", precio_unitario="0.05", base_sin_iva="0.05",
                   iva_tasa="0.1", iva_valor="0.01", valor_total_cotizacion="0.06"),
             valid(cotizacion_id="DATE3", fecha_cotizacion="2026/02/03")]
    audit = run.ingest(put(config, rows))
    assert_balanced(config, audit)
    assert audit["metrics"]["rejected"] == len(invalid)
    main = read(config, "main")
    assert main.filter(F.col("cotizacion_id") == "ALERT").filter(F.size("_np_quality_alerts") >= 5).count() == 1
    n = main.filter(F.col("cotizacion_id") == "NULL").first()
    assert n.descuento_valor is None and n.iva_tasa is None and n._np_quality_alerts
    for key, alerted in [("T01", False), ("T02", True), ("HALF", False)]:
        alert = [a for a in main.filter(F.col("cotizacion_id") == key).first()._np_quality_alerts
                 if a.field == "valor_total_cotizacion"]
        assert bool(alert) == alerted
    q = read(config, "quarantine")
    assert q.filter(F.size("errors") > 0).count() == len(invalid)
    report.update({ca: "passed" for ca in ["CA06", "CA07", "CA08", "CA09", "CA10"]})

def check_conflicts_and_replay():
    config = config_for("replay")
    store, run = engine(config)
    a_content = as_csv([valid(), valid(cotizacion_id="KEEP")])
    source = put(config, content=a_content)
    a = run.ingest(source)
    duplicate_reception = put(config, content=a_content, alternate=True)
    replay = run.ingest(duplicate_reception)
    assert replay["file_id"] == a["file_id"] and len(replay["receipts"]) == 2
    b = run.ingest(put(config, [valid(estado_cotizacion="Rechazada"), valid(cotizacion_id="NEW")]))
    assert b["file_id"] != a["file_id"]
    replay_old = run.ingest(put(config, content=a_content))
    assert read(config, "main").filter(F.col("cotizacion_id") == "Q001").first().estado_cotizacion == "Rechazada"
    assert read(config, "main").count() == 3
    assert read(config, "record_evidence").filter(F.col("_np_file_id") == a["file_id"]).count() == 2
    conflict = run.ingest(put(config, [valid(cotizacion_id="CONFLICT"),
        valid(cotizacion_id="CONFLICT"), valid(cotizacion_id="CONFLICT", cantidad="3")]))
    assert conflict["metrics"] == {"read": 3, "accepted": 0, "rejected": 3, "consolidated": 0}
    assert read(config, "quarantine").filter(F.col("_np_file_id") == conflict["file_id"]).count() == 3
    assert_balanced(config, conflict)
    # January, February, next January all share one main path/FQN.
    stable = cotizaciones_paths(config)["main"]
    for year, month, pk in [("2026", "01", "JAN"), ("2027", "01", "NEXT")]:
        config2 = config | {"data_year": year, "raw_year_path": config["raw_area_root"] + "/" + year}
        _, run2 = engine(config2)
        run2.ingest(put(config2, [valid(cotizacion_id=pk, fecha_id=year + month + "03",
            fecha_cotizacion=year + "-" + month + "-03", mes_carga=year + "-" + month)], month=month))
        assert cotizaciones_paths(config2)["main"] == stable
    assert read(config, "main").count() == 5
    report["CA12"] = "passed"

def check_failures_and_archives():
    for stage in ["ORIGINAL", "EVIDENCE", "MAIN", "QUARANTINE", "AUDIT", "SUCCESS"]:
        config = config_for("failure_" + stage.lower())
        store = FaultStore(spark, dbutils, config, fail_after=stage)
        _, run = engine(config, store)
        source = put(config, [valid(), valid(cotizacion_id="")])
        expect_failure(lambda: run.ingest(source), "INJECTED_")
        assert store.exists(source["source_path"])
        assert all(a["status"] == "FAILED" for a in audit_values(config))
        assert read(config, "failed_files").count() == 1
        audit = run.ingest(source)
        assert_balanced(config, audit)
        assert read(config, "main").count() == 1 and read(config, "quarantine").count() == 1
        assert read(config, "record_evidence").count() == 2
        # Completed replay must not repeat the business MERGE (Delta version stable).
        before = store.DeltaTable.forPath(spark, store.paths["main"]).history(1).first().version
        run.ingest(source)
        after = store.DeltaTable.forPath(spark, store.paths["main"]).history(1).first().version
        assert before == after
    config = config_for("pending")
    store = FaultStore(spark, dbutils, config, fail_after="MAIN")
    _, run = engine(config, store)
    source = put(config, [valid()])
    expect_failure(lambda: run.ingest(source))
    audit_b = run.ingest(put(config, [valid(estado_cotizacion="Vencida")]))
    assert all(a["status"] == "SUCCESS" for a in audit_values(config))
    assert read(config, "main").first().estado_cotizacion == "Vencida"
    # An active reservation rejects a second writer. No automatic stealing.
    store.reserve("test-active-owner")
    try:
        competitor = SparkDeltaStore(spark, dbutils, config)
        expect_failure(lambda: competitor.reserve("other-owner"), "RESERVATION")
    finally:
        store.release("test-active-owner")

    config = config_for("double_log")
    store = FaultStore(spark, dbutils, config, fail_after="MAIN", fail_log=True)
    _, run = engine(config, store)
    source = put(config, [valid()])
    expect_failure(lambda: run.ingest(source), "FAILED_LOG_UNAVAILABLE")
    assert store.exists(source["source_path"])
    assert all(a["status"] == "FAILED" for a in audit_values(config))

    config = config_for("archive")
    store = FaultStore(spark, dbutils, config)
    _, run = engine(config, store)
    source = put(config, [valid(cotizacion_id="")])
    audit = run.ingest(source)
    assert audit["metrics"]["accepted"] == 0 and audit["status"] == "SUCCESS"
    receipt = audit["receipts"][0]
    store.reserve("archive-test")
    try:
        dbutils.fs.put(source["source_path"], "changed", True)
        expect_failure(lambda: archive_receipt(store, audit, receipt), "CHECKSUM")
        dbutils.fs.put(source["source_path"], as_csv([valid(cotizacion_id="")]), True)
        dbutils.fs.mkdirs(receipt["historic_path"].rsplit("/", 1)[0])
        dbutils.fs.put(receipt["historic_path"], "collision", False)
        expect_failure(lambda: archive_receipt(store, audit, receipt), "CHECKSUM")
        # Remove only a known synthetic collision created by this test.
        dbutils.fs.rm(receipt["historic_path"], False)
        store.fail_after = "MOVE"
        expect_failure(lambda: archive_receipt(store, audit, receipt), "INJECTED_MOVE")
        assert not store.exists(source["source_path"]) and store.exists(receipt["historic_path"])
        archive_receipt(store, audit, receipt)
        assert receipt["archived"] and store.checksum(receipt["historic_path"]) == audit["checksum"]
        dbutils.fs.put(source["source_path"], as_csv([valid(cotizacion_id="")]), False)
        archive_receipt(store, audit, receipt)
        assert not store.exists(source["source_path"])
        dbutils.fs.rm(receipt["historic_path"], False)
        expect_failure(lambda: archive_receipt(store, audit, receipt), "MISSING")
    finally:
        store.release("archive-test")
    report["CA13"] = "passed"


In [ ]:
def check_main_integrity_on_replay_and_archive():
    # R1: mutate only this run's explicitly approved synthetic namespace.
    from src.common.cotizaciones_ingestion import archive_cotizaciones
    config = config_for("main_integrity")
    store, run = engine(config)
    source = put(config, [valid(tienda_id="")])
    audit_a = run.ingest(source)
    delta_main = store.DeltaTable.forPath(spark, store.paths["main"])
    condition = F.col("cotizacion_id") == "Q001"
    original_columns = read(config, "main").columns

    def assert_archive_blocked():
        archived, failures = archive_cotizaciones(spark, dbutils, config)
        assert not archived and failures
        assert store.exists(source["source_path"])
        assert not store.exists(audit_a["receipts"][0]["historic_path"])

    # Same sequence, altered scalar: replay and archive must reject it.
    delta_main.update(condition=condition, set={"estado_cotizacion": F.lit("Vencida")})
    expect_failure(lambda: store.reconcile(audit_a, current=False), "MAIN_VALUE_CONCILIATION_FAILED")
    expect_failure(lambda: run.ingest(source), "MAIN_VALUE_CONCILIATION_FAILED")
    assert_archive_blocked()
    delta_main.update(condition=condition, set={"estado_cotizacion": F.lit("Aceptada")})

    # Same sequence, NULL altered into a value: compare with NULL-safe equality.
    delta_main.update(condition=condition, set={"tienda_id": F.lit("inventada")})
    expect_failure(lambda: store.reconcile(audit_a, current=False), "MAIN_VALUE_CONCILIATION_FAILED")
    delta_main.update(condition=condition, set={"tienda_id": F.lit(None).cast("string")})

    # Missing accepted PK cannot be hidden by a surviving/empty Delta table.
    delta_main.delete(condition)
    expect_failure(lambda: run.ingest(source), "MAIN_CONCILIATION_FAILED")
    assert_archive_blocked()
    # Restore exact original from immutable evidence, never from operational data.
    read(config, "record_evidence").filter(
        (F.col("_np_file_id") == audit_a["file_id"]) & (F.col("category") == "ACCEPTED")
    ).select(*original_columns).write.format("delta").mode("append").save(store.paths["main"])
    store.reconcile(audit_a, current=False)

    # Missing sequence is invalid even when business values match.
    delta_main.update(condition=condition, set={"_np_sequence": F.lit(None).cast("long")})
    expect_failure(lambda: store.reconcile(audit_a, current=False), "MAIN_CONCILIATION_FAILED")
    delta_main.update(condition=condition, set={"_np_sequence": F.lit(audit_a["sequence"])})

    # A later real correction is valid; replay A neither rejects B nor rewrites it.
    audit_b = run.ingest(put(config, [valid(tienda_id="", estado_cotizacion="Rechazada")]))
    assert audit_b["sequence"] > audit_a["sequence"]
    store.reconcile(audit_a, current=False)
    version_before = delta_main.history(1).first().version
    run.ingest(put(config, [valid(tienda_id="")]))
    assert read(config, "main").first().estado_cotizacion == "Rechazada"
    assert delta_main.history(1).first().version == version_before
    archived, failures = archive_cotizaciones(spark, dbutils, config)
    assert archived and failures  # B receipt cannot archive the now-replayed A bytes.
    assert not store.exists(source["source_path"])
    report["CA13"] = "passed"


In [ ]:
def check_file_errors():
    for label, content in [("empty", ""), ("header", "id\\n1\\n"), ("framing", '"unclosed\\n')]:
        config = config_for("csv_" + label)
        store, run = engine(config)
        source = put(config, content=content)
        expect_failure(lambda: run.ingest(source))
        assert store.exists(source["source_path"]) and read(config, "failed_files").count() == 1
    config = config_for("extras")
    store, run = engine(config)
    audit = run.ingest(put(config, content=as_csv([valid() | {"extra": "evidencia"}], extra=("extra",))))
    assert "extra" not in read(config, "main").columns
    assert "evidencia" in read(config, "record_evidence").first().payload
    assert any(a.code == "EXTRA_COLUMN" for a in read(config, "main").first()._np_quality_alerts)
    config = config_for("incompatible")
    spark.createDataFrame([("other",)], ["wrong"]).write.format("delta").save(cotizaciones_paths(config)["main"])
    store, run = engine(config)
    expect_failure(lambda: run.ingest(put(config, [valid()])), "INCOMPATIBLE")
    assert read(config, "main").columns == ["wrong"]
    config = config_for("wrongmonth")
    store, run = engine(config)
    source = put(config, [valid()], month="01", name="fact_cotizaciones_02.csv")
    expect_failure(lambda: run.ingest(source), "MISMATCH")
    assert read(config, "failed_files").count() == 1

def check_generic_regression():
    # Only config and taskValues sink are injected; source reads, casts,
    # validation, MERGE and archive run unchanged with real Spark/Delta.
    sink = {}
    proxy = SimpleNamespace(fs=dbutils.fs, jobs=SimpleNamespace(taskValues=SimpleNamespace(
        set=lambda key, value: sink.update({key: json.loads(value)}))))
    for label, area, table in [("generic_comercial", "comercial", "fact_ventas_cabecera"),
                               ("generic_shared", "shared", "dim_tienda")]:
        config = config_for(label, area=area)
        for name in (table, config["audit_table_name"]):
            fqn = test_catalog + "." + config["bronze_schema"] + "." + name
            planned_tables.add(fqn)
            inventory_event("table", fqn, "PLANNED")
        generic_path = assert_test_path(config["bronze_year_root"] + "/" + table, storage_root)
        planned_delta_paths.add(generic_path)
        inventory_event("delta_path", generic_path, "PLANNED")
        dbutils.fs.put(config["data_dictionary_path"],
            "tabla,campo,rol,tipo_dato_csv\n" + table + ",id,PK,object\n" +
            table + ",mes_carga,,object\n", False)
        path = config["raw_year_path"] + "/02/" + table + "_02.csv"
        dbutils.fs.mkdirs(path.rsplit("/", 1)[0])
        dbutils.fs.put(path, "id,mes_carga\nA,2026-02\nB,2026-02\n", False)
        with patch("src.common.notebook_runner.get_config", return_value=config):
            summary = run_bronze_load(spark, proxy, area)
            assert summary["status"] == "OK" and len(summary["processed_files"]) == 1
            assert summary["processed_files"][0]["record_count"] == 2
            actual = spark.table(test_catalog + "." + config["bronze_schema"] + "." + table)
            assert {r.id for r in actual.select("id").toLocalIterator()} == {"A", "B"}
            assert not SparkDeltaStore(spark, dbutils, config)._exists_delta(cotizaciones_paths(config)["file_audit"])
            archived = run_bronze_archive(spark, proxy, area)
            assert archived["status"] == "OK" and len(archived["archived_files"]) == 1
    report["CA14"] = "passed"

def run_all_checks():
    check_contract_and_evidence()
    check_quality()
    check_conflicts_and_replay()
    check_failures_and_archives()
    check_file_errors()
    check_main_integrity_on_replay_and_archive()
    check_generic_regression()
    assert set(report) == {"CA" + str(i).zfill(2) for i in range(1, 15)}
    assert all(value == "passed" for value in report.values())
    result = {"metadata": metadata, "assertions_by_CA": dict(report),
        "cleanup": "No automática; conservar namespace aislado para revisión"}
    print(json.dumps(result, ensure_ascii=False))
    return result

result = None
try:
    result = run_all_checks()
finally:
    finalize_inventory()
# Reached only if every assertion and inventory finalization succeeded.
dbutils.notebook.exit(json.dumps(result, ensure_ascii=False))
